# 🖥️ CPU Benchmark — Matmul FP32 / FP64
Benchmark multiplication de matrices sur CPU (via PyTorch + OpenBLAS/MKL). Mesure les GFLOPS réels par itération, le pic de performance et la moyenne stabilisée.

> **Note :** Le CPU travaille en FP32 ou FP64 (pas FP16 natif comme les Tensor Cores GPU). Les résultats sont en **GFLOPS** (et non TFLOPS) — un CPU haut de gamme atteint ~100–500 GFLOPS, contre ~200+ TFLOPS pour un GPU gaming.

In [1]:
import torch
import time
import os
import platform
import psutil

# ── Infos CPU ─────────────────────────────────────────────────────────────────
device = torch.device("cpu")

cpu_name      = platform.processor() or platform.machine()
cpu_cores     = os.cpu_count()
cpu_freq      = psutil.cpu_freq()
ram_total     = psutil.virtual_memory().total / 1e9
torch_threads = torch.get_num_threads()

# Essayer de récupérer un nom CPU plus lisible
try:
    import subprocess
    if platform.system() == "Windows":
        cpu_name = subprocess.check_output(
            "wmic cpu get name", shell=True
        ).decode().strip().splitlines()[-1].strip()
    elif platform.system() == "Linux":
        with open("/proc/cpuinfo") as f:
            for line in f:
                if "model name" in line:
                    cpu_name = line.split(":", 1)[1].strip()
                    break
    elif platform.system() == "Darwin":
        cpu_name = subprocess.check_output(
            ["sysctl", "-n", "machdep.cpu.brand_string"]
        ).decode().strip()
except Exception:
    pass

freq_str = f"{cpu_freq.current:.0f} MHz" if cpu_freq else "N/A"
print(f"CPU      : {cpu_name}")
print(f"Cœurs    : {cpu_cores}")
print(f"Fréquence: {freq_str}")
print(f"RAM      : {ram_total:.2f} Go")
print(f"Threads PyTorch : {torch_threads}")
print(f"Backend BLAS    : {torch.__config__.show().splitlines()[0]}")

c:\Users\mgraz\Documents\.vscode\Dépot GitHub\Benchmark_CPU_-_GPU\.venv\lib\site-packages\torch\_subclasses\functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


CPU      : AMD Ryzen 5 5600G with Radeon Graphics
Cœurs    : 12
Fréquence: 3901 MHz
RAM      : 16.75 Go
Threads PyTorch : 6
Backend BLAS    : PyTorch built with:


In [2]:
# ── Paramètres ────────────────────────────────────────────────────────────────
# Taille réduite vs GPU : le CPU est ~100-1000x plus lent en matmul dense
MATRIX_SIZE  = 2_000    # N×N — ajuster selon la RAM disponible
DTYPE        = torch.float32   # FP32 (changer en float64 pour DP)
WARMUP_ITERS = 3
BENCH_ITERS  = 10

def get_cpu_util() -> float:
    """Utilisation CPU globale (%), snapshot 0.1 s."""
    try:
        return psutil.cpu_percent(interval=0.1)
    except Exception:
        return float("nan")

def get_ram_used() -> float:
    """RAM utilisée en Go."""
    return psutil.virtual_memory().used / 1e9

def measure_gflops(A: torch.Tensor, B: torch.Tensor) -> tuple[float, float]:
    """Retourne (GFLOPS, elapsed_ms) pour A × B sur CPU."""
    n = A.shape[0]
    t0 = time.perf_counter()
    _ = torch.matmul(A, B)
    t1 = time.perf_counter()
    elapsed_s = t1 - t0
    flops = 2 * n ** 3
    gflops = flops / elapsed_s / 1e9
    return gflops, elapsed_s * 1e3

# ── Allocation ────────────────────────────────────────────────────────────────
dtype_str = str(DTYPE).split(".")[1]
print(f"Allocation de 2 matrices {MATRIX_SIZE}×{MATRIX_SIZE} ({dtype_str}) sur CPU…")
A = torch.randn(MATRIX_SIZE, MATRIX_SIZE, dtype=DTYPE)
B = torch.randn(MATRIX_SIZE, MATRIX_SIZE, dtype=DTYPE)
matrix_ram = 2 * MATRIX_SIZE**2 * (4 if DTYPE == torch.float32 else 8) / 1e9
print(f"RAM matrices : {matrix_ram:.3f} Go")

# ── Warmup ────────────────────────────────────────────────────────────────────
print(f"\nChauffe ({WARMUP_ITERS} itérations)…")
for _ in range(WARMUP_ITERS):
    _ = torch.matmul(A, B)
print("Warmup terminé.")

# ── Benchmark ─────────────────────────────────────────────────────────────────
print(f"\n{'─'*70}")
print(f"{'Iter':>5}  {'GFLOPS':>8}  {'Temps (ms)':>11}  {'CPU Util':>9}  {'RAM':>8}")
print(f"{'─'*70}")

gflops_history = []
elapsed_history = []
start_total = time.perf_counter()

for it in range(1, BENCH_ITERS + 1):
    gflops, ms = measure_gflops(A, B)
    gflops_history.append(gflops)
    elapsed_history.append(ms)

    util = get_cpu_util()
    ram  = get_ram_used()
    util_str = f"{util:.0f}%" if util == util else "N/A"
    print(f"{it:>5}  {gflops:>8.2f}  {ms:>11.1f}  {util_str:>9}  {ram:>6.2f} Go")

total_elapsed = time.perf_counter() - start_total
print(f"{'─'*70}")

# ── Statistiques ──────────────────────────────────────────────────────────────
peak_gflops   = max(gflops_history)
mean_gflops   = sum(gflops_history) / len(gflops_history)
stable_mean   = sum(gflops_history[1:]) / len(gflops_history[1:])
mean_ms       = sum(elapsed_history) / len(elapsed_history)

print(f"\n{'═'*70}")
print(f"  Benchmark terminé en {total_elapsed:.2f}s ({BENCH_ITERS} itérations)")
print(f"  Matrice : {MATRIX_SIZE}×{MATRIX_SIZE} {dtype_str} | {torch_threads} threads PyTorch")
print(f"{'─'*70}")
print(f"  GFLOPS peak       : {peak_gflops:.2f}")
print(f"  GFLOPS moyen      : {mean_gflops:.2f}")
print(f"  GFLOPS stabilisé  : {stable_mean:.2f}  (iter 2–{BENCH_ITERS})")
print(f"  Temps moyen/iter  : {mean_ms:.1f} ms")
print(f"{'═'*70}")

Allocation de 2 matrices 2000×2000 (float32) sur CPU…
RAM matrices : 0.032 Go

Chauffe (3 itérations)…
Warmup terminé.

──────────────────────────────────────────────────────────────────────
 Iter    GFLOPS   Temps (ms)   CPU Util       RAM
──────────────────────────────────────────────────────────────────────
    1    260.97         61.3        60%   13.30 Go
    2    281.68         56.8        63%   13.30 Go
    3    278.67         57.4        51%   13.30 Go
    4    306.64         52.2        51%   13.30 Go
    5    285.42         56.1        57%   13.30 Go
    6    293.23         54.6        60%   13.30 Go
    7    268.51         59.6        55%   13.30 Go
    8    304.99         52.5        76%   13.31 Go
    9    296.91         53.9        50%   13.30 Go
   10    287.36         55.7        58%   13.30 Go
──────────────────────────────────────────────────────────────────────

══════════════════════════════════════════════════════════════════════
  Benchmark terminé en 1.66s (10 it

In [3]:
# ── Bonus : scaling selon le nombre de threads ─────────────────────────────────
# Montre comment les GFLOPS évoluent de 1 thread jusqu'au max disponible
import math

max_threads = torch.get_num_threads()
thread_counts = sorted(set(
    [1, 2, 4] +
    [2**i for i in range(3, int(math.log2(max_threads)) + 1) if 2**i <= max_threads] +
    [max_threads]
))

print("Scaling selon le nombre de threads PyTorch…")
print(f"{'─'*40}")
print(f"{'Threads':>8}  {'GFLOPS':>8}  {'Speedup':>8}")
print(f"{'─'*40}")

thread_results = []
baseline = None

for n_threads in thread_counts:
    torch.set_num_threads(n_threads)
    # Warmup rapide
    _ = torch.matmul(A, B)
    # Moyenne sur 3 runs
    g_vals = [measure_gflops(A, B)[0] for _ in range(3)]
    g = sum(g_vals) / len(g_vals)
    if baseline is None:
        baseline = g
    speedup = g / baseline
    thread_results.append((n_threads, g))
    print(f"{n_threads:>8}  {g:>8.2f}  {speedup:>7.2f}x")

# Restaurer le nombre de threads original
torch.set_num_threads(max_threads)
print(f"{'─'*40}")
print(f"Threads restaurés : {max_threads}")

Scaling selon le nombre de threads PyTorch…
────────────────────────────────────────
 Threads    GFLOPS   Speedup
────────────────────────────────────────
       1     88.17     1.00x
       2    157.78     1.79x
       4    213.86     2.43x
       6    291.80     3.31x
────────────────────────────────────────
Threads restaurés : 6


In [4]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Graphique 1 : GFLOPS par itération ────────────────────────────────────────
ax = axes[0]
iters = list(range(1, len(gflops_history) + 1))
ax.bar(iters, gflops_history, color="#4A90D9", alpha=0.85, zorder=3, label="GFLOPS par itération")
ax.axhline(mean_gflops,  color="#E85D24", linestyle="--", linewidth=1.5, label=f"Moyenne   : {mean_gflops:.1f} GFLOPS")
ax.axhline(stable_mean,  color="#2DBD6E", linestyle=":",  linewidth=1.5, label=f"Stabilisé : {stable_mean:.1f} GFLOPS")
ax.axhline(peak_gflops,  color="#FAC41A", linestyle="-.", linewidth=1.2, label=f"Peak      : {peak_gflops:.1f} GFLOPS")
ax.set_xlabel("Itération", fontsize=11)
ax.set_ylabel("GFLOPS (FP32)", fontsize=11)
ax.set_title(f"Matmul {MATRIX_SIZE}×{MATRIX_SIZE} — {BENCH_ITERS} itérations", fontsize=11, fontweight="bold")
ax.set_xticks(iters)
ax.yaxis.set_minor_locator(mtick.AutoMinorLocator())
ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
ax.legend(fontsize=8, loc="lower right")
ax.set_ylim(0, peak_gflops * 1.2)

# ── Graphique 2 : scaling threads ─────────────────────────────────────────────
ax2 = axes[1]
t_x = [r[0] for r in thread_results]
t_y = [r[1] for r in thread_results]
ideal_y = [t_y[0] * (n / t_x[0]) for n in t_x]

ax2.plot(t_x, t_y,     "o-", color="#4A90D9", linewidth=2, markersize=6, label="Mesuré")
ax2.plot(t_x, ideal_y, "--", color="#ccc",    linewidth=1.2, label="Scaling idéal (linéaire)")
ax2.fill_between(t_x, t_y, ideal_y, alpha=0.08, color="#E85D24", label="Écart vs idéal")
ax2.set_xlabel("Nombre de threads", fontsize=11)
ax2.set_ylabel("GFLOPS (FP32)", fontsize=11)
ax2.set_title("Scaling multi-thread", fontsize=11, fontweight="bold")
ax2.set_xticks(t_x)
ax2.yaxis.set_minor_locator(mtick.AutoMinorLocator())
ax2.grid(linestyle="--", alpha=0.4)
ax2.legend(fontsize=8)
ax2.set_ylim(0, max(ideal_y) * 1.15)

fig.suptitle(f"CPU Benchmark — {cpu_name}", fontsize=12, fontweight="bold", y=1.02)
fig.tight_layout()
plt.savefig("cpu_benchmark_result.png", dpi=150, bbox_inches="tight")
plt.show()
print("Graphique sauvegardé → cpu_benchmark_result.png")

ModuleNotFoundError: No module named 'matplotlib'